## 딥페이크 범죄 대응을 위한 AI 탐지 모델 경진대회

**※주의** : 반드시 본 파일을 이용하여 제출을 수행해야 하며, 파일의 이름은 `task.ipynb`로 유지되어야 합니다.

* #### 추론 실행 환경
    * `python 3.9` 환경
    * `CUDA 10.2`, `CUDA 11.8`, `CUDA 12.6`를 지원합니다.
    * 각 CUDA 환경에 미리 설치돼있는 torch 버전은 다음 표를 참고하세요.

<table>
  <thead>
    <tr>
      <th align="center">Python</th>
      <th align="center">CUDA</th>
      <th align="center">torch</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="center" style="vertical-align: middle;">3.8</td>
      <td align="center">10.2</td>
      <td align="center">1.6.0</td>
    </tr>
    <tr>
      <td align="center" style="vertical-align: middle;">3.9</td>
      <td align="center">11.8</td>
      <td align="center">1.8.0</td>
    </tr>
    <tr>
      <td align="center">3.10</td>
      <td align="center">12.6</td>
      <td align="center">2.7.1</td>
    </tr>
  </tbody>
</table>

* #### CUDA 버전 관련 안내사항  
  - 이번 경진대회는 3개의 CUDA 버전을 지원합니다.  
  - 참가자는 자신의 모델의 라이브러리 의존성에 맞는 CUDA 환경을 선택하여 모델을 제출하면 됩니다.   
  - 각 CUDA 환경에는 기본적으로 torch가 설치되어 있으나, 참가자는 제출하는 CUDA 버전과 호환되는 torch, 필요한 버전의 라이브러리를 `!pip install` 하여 사용하여도 무관합니다.

* #### `task.ipynb` 작성 규칙
코드는 크게 3가지 파트로 구성되며, 해당 파트의 특성을 지켜서 내용을 편집하세요.   
1. **제출용 aifactory 라이브러리 및 추가 필요 라이브러리 설치**
    - 채점 및 제출을 위한 aifactory 라이브러리를 설치하는 셀입니다. 이 부분은 수정하지 않고 그대로 실행합니다.
    - 그 외로, 모델 추론에 필요한 라이브러리를 직접 설치합니다.
2. **추론용 코드 작성**
    - 모델 로드, 데이터 전처리, 예측 등 실제 추론을 수행하는 모든 코드를 이 영역에 작성합니다.
3. **aif.submit() 함수를 호출하여 최종 결과를 제출**
    - **마이 페이지-활동히스토리**에서 발급받은 key 값을 함수의 인자로 정확히 입력해야 합니다.
    - **※주의** : 제출하고자 하는 CUDA 환경에 맞는 key를 입력하여야 합니다.

<table>
  <thead>
    <tr>
      <th align="left">Competition 이름</th>
      <th align="center">CUDA</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="left">딥페이크 범죄 대응을 위한 AI 탐지 모델 경진대회</td>
      <td align="center">11.8</td>
    </tr>
    <tr>
      <td align="left">딥페이크 범죄 대응을 위한 AI 탐지 모델 경진대회 CUDA 12.6</td>
      <td align="center">12.6</td>
    </tr>
    <tr>
      <td align="left">딥페이크 범죄 대응을 위한 AI 탐지 모델 경진대회 CUDA 10.2</td>
      <td align="center">10.2</td>
    </tr>
  </tbody>
</table>

------

#### 1. 제출용 aifactory 라이브러리 설치
※ 결과 전송에 필요하므로 아래와 같이 aifactory 라이브러리가 반드시 최신버전으로 설치될 수 있게끔 합니다

In [1]:
!pip install -U aifactory

* 자신의 모델 추론 실행에 필요한 추가 라이브러리 설치

In [2]:
!pip install transformers
!pip install -U torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -U datasets
!pip install -U opencv-python==4.10.0.82 numpy==1.26.4 scikit-learn==1.3.2 scipy==1.11.4
!pip install -U pathlib
!pip install -U dlib
!pip install open_clip_torch
!pip install tqdm

Looking in indexes: https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement torch==2.7.1 (from versions: none)
ERROR: No matching distribution found for torch==2.7.1
  Using cached opencv_python-4.10.0.82-cp37-abi3-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Installing backend dependencies ... one
done
  Using cached scikit-learn-1.3.2.tar.gz (7.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [668 lines of output]
      Partial import of sklearn during the build process.
      clang: error: unsupported option '-fopenmp'
      /private/var/folders/x0/bvf92wl546n44r4b4x5q4yd40000gn/T/pip-install-36ek

-----

#### 2. 추론용 코드 작성

##### 추론 환경의 기본 경로 구조

- 평가 데이터셋 경로: `./data/`
   - 채점에 사용될 테스트 데이터셋은 `./data/` 디렉토리 안에 포함되어 있습니다.
   - 해당 디렉토리에는 이미지(JPG, PNG)와 동영상(MP4) 파일이 별도의 하위 폴더 없이 혼합되어 있습니다.
```bash
/aif/
└── data/
    ├── {이미지 데이터1}.jpg
    ├── {이미지 데이터2}.png
    ├── {동영상 데이터1}.mp4
    ├── {이미지 데이터3}.png
    ├── {동영상 데이터2}.mp4
    ...
```

- 모델 및 자원 경로: 예시 : `./model/`
   - 추론 스크립트가 실행되는 위치를 기준으로, 제출된 모델 관련 파일들이 위치해야하 하는 상대 경로입니다.
   - 학습된 모델 가중치(.pt, .ckpt, .pth 등)

* 제출 파일은 `submission.csv`로 저장돼야 합니다.
  * submission.csv는 *filename*과 *label* 컬럼으로 구성돼야 합니다.
  * filename은 추론한 파일의 이름(확장자 포함), label은 추론 결과입니다. (real:0, fake:1)
  * filename은 *string*, label은 *int* 자료형이어야 합니다.
  * 추론하는 데이터의 순서는 무작위로 섞여도 상관 없습니다.

<table>
  <thead>
    <tr>
      <th align="center">filename</th>
      <th align="center">label</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="center">{이미지 데이터1}.jpg</td>
      <td align="center">0</td>
    </tr>
    <tr>
      <td align="center">{동영상 데이터1}.mp4</td>
      <td align="center">1</td>
    </tr>
    <tr>
      <td colspan="2" align="center">...</td>
    </tr>
  </tbody>
</table>

**※ 주의 사항**

* argparse 사용시 `args, _ = parser.parse_known_args()`로 인자를 지정하세요.   
   - `args = parser.parse_args()`는 jupyter에서 오류가 발생합니다.
* return 할 결과물과 양식에 유의하세요.

In [3]:
import os
os.system("unzip submission_bundle.zip")
os.system("ls")

model
task.ipynb


unzip:  cannot find or open submission_bundle.zip, submission_bundle.zip.zip or submission_bundle.zip.ZIP.


0

In [ ]:
import os, csv, cv2, dlib, torch, multiprocessing, pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import open_clip

device = "cuda"

# ==============================================
# 1) SRM Branch
# ==============================================
class SRMConv(nn.Module):
    def __init__(self):
        super().__init__()
        kernel = torch.tensor([
            [-1,-1,-1],
            [-1, 8,-1],
            [-1,-1,-1]
        ], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        self.register_buffer("weight", kernel)

    def forward(self, x):
        gray = x.mean(dim=1, keepdim=True)
        return F.conv2d(gray, self.weight, padding=1)


class Deepfake_ViTH14_SRMBRANCH(nn.Module):
    def __init__(self, backbone, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.srm = SRMConv()
        self.fuse = nn.Conv2d(4, 3, kernel_size=1)
        hidden = backbone.visual.output_dim
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):
        srm = self.srm(x)
        x = torch.cat([x, srm], dim=1)
        x = self.fuse(x)
        feats = self.backbone.encode_image(x)
        return self.fc(feats)


# ==============================================
# 2) Load Model + Preprocess (LOCAL ONLY)
# ==============================================
save_root = "./model"

with open(f"{save_root}/preprocess.pkl", "rb") as f:
    preprocess = pickle.load(f)

backbone, _, _ = open_clip.create_model_and_transforms("ViT-H-14", pretrained=None)
backbone = backbone.to(device)

model = Deepfake_ViTH14_SRMBRANCH(backbone).to(device)
model.load_state_dict(torch.load(f"{save_root}/ViT-H14_SRMBRANCH.pth", map_location=device))
model.eval()

print("✅ Model Loaded")


# ==============================================
# 3) Face crop util (주최측 그대로 유지)
# ==============================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
VIDEO_EXTS = {".avi", ".mp4"}

def get_boundingbox(face, width, height):
    x1, y1, x2, y2 = face.left(), face.top(), face.right(), face.bottom()
    size_bb = int(max(x2 - x1, y2 - y1) * 1.3)
    center_x, center_y = (x1 + x2)//2, (y1 + y2)//2
    x1 = max(int(center_x - size_bb//2), 0)
    y1 = max(int(center_y - size_bb//2), 0)
    size_bb = min(width-x1, size_bb)
    size_bb = min(height-y1, size_bb)
    return x1, y1, size_bb

def detect_and_crop(image):
    detector = dlib.get_frontal_face_detector()
    np_img = np.array(image)
    faces = detector(np_img, 1)
    if not faces:
        return None
    face = max(faces, key=lambda r: r.width()*r.height())
    x,y,s = get_boundingbox(face, np_img.shape[1], np_img.shape[0])
    face_img = np_img[y:y+s, x:x+s]
    return Image.fromarray(face_img).resize((224,224), Image.BICUBIC)


# ==============================================
# 4) Inference Loop
# ==============================================
test_dataset_path = Path("./data")
output_csv_path = Path("submission.csv")

results = {}
files = [p for p in sorted(test_dataset_path.iterdir()) if p.is_file()]

for file in tqdm(files):
    try:
        img = Image.open(file).convert("RGB")
        face = detect_and_crop(img)
        if face is None:
            results[file.name] = 0
            continue
        t = preprocess(face).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(t)
            pred = logits.argmax(1).item()
        results[file.name] = pred
    except:
        results[file.name] = 0

with open(output_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "label"])
    for k,v in results.items():
        writer.writerow([k,v])

print("✅ Submission saved to submission.csv")



✅ Model loaded: ./model/CLIP-ViT-B16_2.pth


FileNotFoundError: [Errno 2] No such file or directory: 'data'

----

#### 3. `aif.submit()` 함수를 호출하여 최종 결과를 제출

**※주의** : task별, 참가자별로 key가 다릅니다. 잘못 입력하지 않도록 유의바랍니다.
- key는 대회 페이지 [베이스라인 코드](https://aifactory.space/task/9197/baseline) 탭에 기재된 가이드라인을 따라 task 별로 확인하실 수 있습니다.
- key가 틀리면 제출이 진행되지 않거나 잘못 제출되므로 task에 맞는 자신의 key를 사용해야 합니다.
-  **NOTE** : 이번 경진대회에서는 3개의 CUDA 버전을 지원하며, 각 CUDA 버전에 따라 task key가 상이합니다. 함수를 실행하기 전에 현재 key가 제출하고자 하는 CUDA 환경에 대한 key인지 반드시 확인하세요.

In [ ]:
import aifactory.score as aif
import time
t = time.time()

#-----------------------------------------------------#
aif.submit(model_name="ViT-H14",
    key="48fdaefc-fed2-48e1-a290-7099d47dfc0b"

)
#-----------------------------------------------------#
print(time.time() - t)

file : fileId=1jDmoCXBR9mCS5p66f_qzv-dH9z43uRHz
Running on CoLab
google colab


Downloading...
From: https://drive.google.com/uc?id=1jDmoCXBR9mCS5p66f_qzv-dH9z43uRHz
To: /content/task.ipynb
100%|██████████| 43.5k/43.5k [00:00<00:00, 47.3MB/s]


제출 완료
341.79898166656494
